### Objetivo
El objetivo es diseñar e implementar un agente inteligente basado en LangGraph capaz de clasificar una consulta textual y asignarla a una especialidad médica adecuada, devolviendo además el teléfono de contacto correspondiente.

El sistema recibe como entrada un texto libre que simula un correo electrónico, una llamada telefónica o un mensaje informal de un paciente, donde se describen síntomas, malestar o la necesidad de contactar con un servicio sanitario. A partir de esta entrada, el agente debe analizar la información, decidir si existe una correspondencia clara con alguna especialidad y, en caso afirmativo, proporcionar los datos de contacto adecuados.

Se proporcionan datos a traves de un archivo .csv que contiene un listado de especialidades y sintomas , asignadas a una descripción y telefono.

### Diseño del Agente
Se ha diseñado un agentic workflow que combina:

- Razonamiento con LLMs, utilizados para:
    - resumir y normalizar la descripción inicial del paciente,
    - validar si una especialidad propuesta es coherente con el caso descrito.

- Recuperación de información (RAG) mediante embeddings, utilizando un conjunto de datos estructurado en formato CSV que contiene:
    - especialidad médica,
    - descripción del servicio,
    - síntomas comunes,
    - teléfono de contacto.

- Lógica de decisión explícita, implementada mediante LangGraph, que controla el paso entre las distintas fases del proceso.

El flujo del agente se organiza en dos etapas principales de análisis:
- 1.- Análisis por descripción del servicio, donde se intenta asignar la consulta a una especialidad comparando el resumen generado por el LLM con las descripciones institucionales de los servicios.

- 2.- Análisis por síntomas, que se activa únicamente cuando la asignación anterior es rechazada o presenta baja coherencia semántica, comparando la consulta con los síntomas comunes asociados a cada especialidad.

Si ninguna de las dos etapas produce una asignación válida, el agente rechaza la solicitud, devolviendo un mensaje explicativo que indica que no se ha encontrado una especialidad adecuada con suficiente similitud.

### Resultado esperado
Como resultado final, el agente devuelve:
- la especialidad médica recomendada,
- el teléfono de contacto asociado,
- o, en caso de no existir una coincidencia suficiente, un mensaje de rechazo justificando la decisión.


### Autores 
 - Ángel Ortiz de Lejarazu Sánchez
 - Ruben Castañeda Matute


In [ ]:
# --- Imports Básicos ---
import os
import pandas as pd # MAnejo CSV
from typing import TypedDict, Optional

# LLM para resumir y validar y embeddings para búsqueda por similitud
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# Plantillas de prompts reutilizables
from langchain_core.prompts import ChatPromptTemplate
# Estructuras pydantic para outputs estructurados
from pydantic import BaseModel, Field

# Para textos
from langchain_core.documents import Document
#Vectorstore en memoria para búsqueda por similitud
from langchain_community.vectorstores import DocArrayInMemorySearch

# Construcción y finalización de grafos de estados
from langgraph.graph import StateGraph, END

In [ ]:
# Manejo de variables de entorno
from dotenv import load_dotenv
import os

# Se usa clave de uso privado, esta cargada en el repositorio porque es un 
# repositorio privado que solo ve el profesor.

load_dotenv('OPENAI.env')  # Carga el archivo OPENAI.env
print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY"))


In [ ]:
#llm = ChatOpenAI(model='gpt-5-nano', model_provider='openai')
llm = ChatOpenAI(model_name='gpt-5-nano')  # o llm = ChatOpenAI(model='gpt-5-nano')

# OpenAIEmbeddings: convierte texto en vectores numéricos (embeddings) 
# para búsquedas semánticas y comparaciones de similitud.
emb = OpenAIEmbeddings()

In [ ]:
#Path local, la misma carpeta donde está este notebook dentro de mi PC
path="./"
df_services=pd.read_csv(f"{path}/sintomas.csv", sep=";")
#df_services.head()

#Listas para almacenar documentos de descripción y síntomas
docs_desc, docs_symp = [], []

#Recorre el DataFrame y crea Documentos con la descripción y los síntomas comunes
for i in df_services.index:
    meta = {
        "row_id": int(i),
        "especialidad": str(df_services.loc[i, "especialidad"]),
        "telefono": str(df_services.loc[i, "telefono"]),
    }
    #Documentos para busqueda por descripción y por síntomas
    docs_desc.append(Document(page_content=str(df_services.loc[i, "descripcion"]), metadata=meta))
    docs_symp.append(Document(page_content=str(df_services.loc[i, "sintomas_comunes"]), metadata=meta))

# Crea dos vectores de búsqueda en memoria, uno por descripción y otro por síntomas
# Es donde se buscará.
# En DocArrayInMemorySearch, el embedding se calcula solo sobre page_content
vdb_desc = DocArrayInMemorySearch.from_documents(docs_desc, embedding=emb)
vdb_symp = DocArrayInMemorySearch.from_documents(docs_symp, embedding=emb)


2)  Agente 

In [ ]:
# AgentState define el “estado compartido” que va pasando por los nodos del grafo. 
# Cada clave guarda un dato del flujo: entrada del usuario (raw_text), resumen del LLM, resultados de las búsquedas
# RAG por descripción y síntomas, la decisión final y los umbrales de similitud. 
# total=False permite que no todas las claves existan desde el inicio; se van añadiendo conforme el flujo avanza.
# ------------------------------------------------------------------------------------------
# Estado del AGENTE (memoria compartida entre nodos del grafo)
class AgentState(TypedDict, total=False):
    # Entrada original del usuario (paciente)
    raw_text: str
    # Resumen generado por el LLM (reviewer)       
    summary: Optional[str]   

    # Resultado RAG descripción
    service_name_desc: Optional[str]
    phone_desc: Optional[str]
    score_desc: Optional[float]

    # Resultado del RAG por sintomas
    service_name_symp: Optional[str]
    phone_symp: Optional[str]
    score_symp: Optional[float]

    # Decision final
    status: Optional[str]       # "aprobado" / "rechazado"
    decision: Optional[str]     # justificación del LLM
    final_service: Optional[str]
    final_phone: Optional[str]
    final_message: Optional[str]

    # Configuracion de umbrales
    threshold_desc: float
    threshold_symp: float



5) Prompts (Reviewer + Validator)

In [ ]:
# Reviewer que resume los sintomas o lo que necesita

REVIEWER_PROMPT = ChatPromptTemplate.from_template("""
Eres un asistente sanitario que resume y normaliza mensajes de pacientes.
Tu tarea:
- Extraer motivo principal, síntomas y contexto.
- Mantenerlo breve (máximo 2 frases).
- No inventes datos.

Mensaje del paciente:
{raw_text}

Devuelve SOLO el resumen.
""")

In [ ]:
# Validator: decide si la especialidad propuesta encaja (aprobado/rechazado) con justificación

class Decision(BaseModel):
    # Estado de la validacion: aprobado o rechazado
    status: str = Field(description="aprobado o rechazado")
    # Explicacion corta de la decision
    decision: str = Field(description="justificación breve (máximo 2 frases)")

VALIDATOR_PROMPT = ChatPromptTemplate.from_template("""
Eres un asistente que valida si la especialidad médica asignada es adecuada.

Tienes:
- Resumen del caso del paciente
- Especialidad propuesta y su descripción

Tarea:
1) Decide si la asignación es coherente.
2) Si NO lo es, status="rechazado".
3) Si SÍ lo es, status="aprobado".
4) Da una justificación breve (máximo 2 frases).

Caso (resumen): {summary}

Especialidad propuesta: {service_name}
Descripción de la especialidad: {service_description}

Devuelve un JSON con:
- status
- decision
""")


### Nodos del grafo

In [ ]:
# LLM “Reviewer”: resume/normaliza lo que dice la persona (correo/llamada/nota).
def reviewer(state: AgentState) -> AgentState:
    # Construye el prompt con el texto original
    msg = REVIEWER_PROMPT.format_messages(raw_text=state["raw_text"])
    # Llama al LLM
    resp = llm.invoke(msg)
    # Devuelve el resumen limpio para el estado  
    return {"summary": resp.content.strip()}

In [ ]:
# Retriever (RAG) por descripción: busca la especialidad más parecida.

def retrieve_desc(state: AgentState) -> AgentState:
    # Log 
    print("Analizando descripcion del problema")

    # Busca el documento mas similar usando el resumen del paciente
    docs = vdb_desc.similarity_search_with_score(query=state["summary"], k=1)
    if not docs:
        # Si no hay resultados, devuelve campos vacios
        return {"service_name_desc": None, "phone_desc": None, "score_desc": None,
                "service_description_desc": None}

    # El mejor resultado (documento + score)
    doc, score = docs[0]
    return {
        # Metadatos de la especialidad 
        "service_name_desc": doc.metadata.get("especialidad"),
        "phone_desc": doc.metadata.get("telefono"),
        # similitud
        "score_desc": float(score),
        # descripcion de la especialidad
        "service_description_desc": doc.page_content,
    }


In [ ]:
# LLM “Validator”: decide si esa especialidad es correcta (aprobado/rechazado) y da justificación.
# Si rechazado → Retriever por síntomas → LLM Validator de nuevo.
# Si vuelve a rechazado → mensaje final de rechazo explicando que no hay similitud suficiente


def validate_desc(state: AgentState) -> AgentState:
    # si no hay candidato por descripción
    if not state.get("service_name_desc"):
        print("No se encontró ninguna especialidad candidata a partir de la descripción.")        
        return {
            "status": "rechazado",
            "decision": "No se encontró ninguna especialidad candidata a partir de la descripción."
        }

# Construye el prompt con resumen y descripcion de la especialidad
    msg = VALIDATOR_PROMPT.format_messages(
        summary=state.get("summary", ""),
        service_name=state.get("service_name_desc", ""),
        service_description=state.get("service_description_desc", "")
    )

    # Llama al LLM y parsea la salida al modelo Decision
    resp = llm.with_structured_output(Decision).invoke(msg)
    status = resp.status.strip().lower()
    decision = resp.decision.strip()

    # Si aprobado: completa salida final y termina
    if status == "aprobado":
        return {
            "status": status,
            "decision": decision,
            "final_service": state.get("service_name_desc"),
            "final_phone": state.get("phone_desc"),
            "final_message": f"Especialidad: {state.get('service_name_desc')}. Teléfono: {state.get('phone_desc')}."
        }

    # Si rechazado: retorna solo status y decision (irá a retrieve_symp)
    return {"status": status, "decision": decision}

In [ ]:
#Condicional Si aprobado termina, sino va a los sintomas.
def route_after_desc(state: AgentState) -> str:
    if state.get("status") == "aprobado":
        return "end"
    return "symptoms"


In [ ]:
# Retriever (RAG) por síntomas: busca la especialidad más parecida.
def retrieve_symp(state: AgentState) -> AgentState:
    # Si ya se aprobó por descripción, no hacer la búsqueda de síntomas
    if state.get("status") == "aprobado":
        return {}
    #log
    print("Analizando Síntomas")
    
    # Busca el documento mas similar usando el resumen del paciente
    docs = vdb_symp.similarity_search_with_score(query=state["summary"], k=1)
    if not docs:
        # Si no hay resultados, devuelve campos vacios
        return {"service_name_symp": None, "phone_symp": None, 
                "score_symp": None, 
                "service_description_symp": None}
    # Toma el mejor resultado (documento + score)
    doc, score = docs[0]
    return {
        # Metadatos de la especialidad encontrada
        "service_name_symp": doc.metadata.get("especialidad"),
        "phone_symp": doc.metadata.get("telefono"),
        # similitud
        "score_symp": float(score),
        # Texto de sintomas
        "service_description_symp": doc.page_content,
    }

In [ ]:
# Validator para sintomas (LLM): valida por sintomas o rechaza

def validate_symp_or_reject(state: AgentState) -> AgentState:
    # Si no hay candidato por sintomas, rechaza directamente
    if not state.get("service_name_symp"):
        return {
            "status": "rechazado",
            "decision": "No se encontró ninguna especialidad candidata a partir de los síntomas.",
            "final_message": "Rechazado: no hay coincidencia suficiente ni por descripción ni por síntomas."
        }

    # Construye el prompt con el resumen y los sintomas recuperados
    msg = VALIDATOR_PROMPT.format_messages(
        summary=state.get("summary", ""),
        service_name=state.get("service_name_symp", ""),
        service_description=state.get("service_description_symp", "")
    )

    # Llama al LLM y parsea la salida al modelo Decision
    resp = llm.with_structured_output(Decision).invoke(msg)

    # Si aprobado: completa salida final
    if resp.status.strip().lower() == "aprobado":
        return {
            "status": "aprobado",
            "decision": resp.decision.strip(),
            "final_service": state.get("service_name_symp"),
            "final_phone": state.get("phone_symp"),
            "final_message": f"Especialidad: {state.get('service_name_symp')}. Teléfono: {state.get('phone_symp')}."
        }

    # Si rechazado: mensaje final
    return {
        "status": "rechazado",
        "decision": resp.decision.strip(),
        "final_message": "Rechazado: la especialidad propuesta no encaja y la similitud fue insuficiente también con síntomas."
    }

In [ ]:
# Nodo final si se aprobó por descripción
# Completa la salida final con la especialidad y telefono

def finalize_if_desc_approved(state: AgentState) -> AgentState:
    return {
        "final_service": state["service_name_desc"],
        "final_phone": state["phone_desc"],
        "final_message": f"Especialidad: {state['service_name_desc']}. Teléfono: {state['phone_desc']}."
    }

In [ ]:
# Construcción del GRAFO
g = StateGraph(AgentState)

# Registro de nodos  del flujo
g.add_node("Reviewer", reviewer)
g.add_node("RetrieveDesc", retrieve_desc)
g.add_node("ValidateDesc", validate_desc)
g.add_node("FinalizeDesc", finalize_if_desc_approved)
g.add_node("RetrieveSymp", retrieve_symp)
g.add_node("ValidateSympOrReject", validate_symp_or_reject)

# Punto de entrada del flujo
g.set_entry_point("Reviewer")

# Flujo principal: resumen -> retrieval por descripcion -> validacion
g.add_edge("Reviewer", "RetrieveDesc")
g.add_edge("RetrieveDesc", "ValidateDesc")

# Enrutado condicional segun la validacion por descripcion
g.add_conditional_edges(
    "ValidateDesc",
    route_after_desc,
    {
        "end": "FinalizeDesc",
        "symptoms": "RetrieveSymp"
    }
)

# Si se aprueba por descripcion, se finaliza
g.add_edge("FinalizeDesc", END)

# Si se rechaza, se busca por sintomas y se valida de nuevo
g.add_edge("RetrieveSymp", "ValidateSympOrReject")
g.add_edge("ValidateSympOrReject", END)

# Compila el grafo en un workflow ejecutable
workflow = g.compile()

In [ ]:
# DIBUJAR EL GRAFO

from IPython.display import Image, SVG

# Opción 1: ASCII (siempre funciona)
#print(workflow.get_graph().draw_ascii())

svg = workflow.get_graph().draw_mermaid_png()
Image(svg)


In [ ]:
# Ejemplo de invocación del workflow completo
raw = "Hola, llevo tres días con fiebre, tos y dolor de garganta. ¿Con quién tengo que hablar?"

out = workflow.invoke({
    "raw_text": raw,
    "threshold_desc": 0.75,
    "threshold_symp": 0.72
})

print("Resumen:", out.get("summary"))
print("Estado:", out.get("status"))
print("Decisión:", out.get("decision"))
print("Final:", out.get("final_message"))

In [ ]:
# Eejemplo de invocación del workflow completo
raw = (
    "Desde ayer me encuentro muy mal. Tengo fiebre alta, escalofríos, "
    "dolor muscular, dolor de garganta y mucha debilidad. "
    "No sé qué me pasa."
)

out = workflow.invoke({
    "raw_text": raw,
    "threshold_desc": 0.75,
    "threshold_symp": 0.72
})

print("Resumen:", out.get("summary"))
print("Estado:", out.get("status"))
print("Decisión:", out.get("decision"))
print("Final:", out.get("final_message"))

In [ ]:
# Ejemplo de workflow que no pasa por síntomas
raw = (
"Hola, quiero saber a qué número tengo que llamar para una consulta médica general."
)

out = workflow.invoke({
    "raw_text": raw,
    "threshold_desc": 0.75,
    "threshold_symp": 0.72
})

print("Resumen:", out.get("summary"))
print("Estado:", out.get("status"))
print("Decisión:", out.get("decision"))
print("Final:", out.get("final_message"))

In [ ]:
# Ejemplo rechazado por descripción y sintomas   
raw = (
"Hola, quiero saber cual es el resultado del partido de ayer entre el Real Madrid y el Valencia."
)

out = workflow.invoke({
    "raw_text": raw,
    "threshold_desc": 0.75,
    "threshold_symp": 0.72
})

print("Resumen:", out.get("summary"))
print("Estado:", out.get("status"))
print("Decisión:", out.get("decision"))
print("Final:", out.get("final_message"))